# Submissao 1 - Logistic Regression
Notebook limpo para inferencia final: le CSV, preprocessa, carrega modelo e guarda submissao.

In [1]:
import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd

from src.data_processing import clean_text
from src.bag_of_words import NumpyBagOfWords
from src.models_numpy.logistic_regression import LogisticRegression

In [2]:
input_csv = Path('../submissions/submissao1/subm1.csv')
train_csv = Path('../data/processed/dataset_kaggle_daigt_processed.csv')

model_candidates = [
    Path('../submissions/submissao1/logistic_regression_final.npz'),
    Path('../saved_models/logistic_regression_final.npz')
]
bow_candidates = [
    Path('../submissions/submissao1/logistic_bow_model.pkl'),
    Path('../saved_models/logistic_bow_model.pkl')
]

model_path = next((p for p in model_candidates if p.exists()), None)
bow_path = next((p for p in bow_candidates if p.exists()), None)

output_dir = Path('../submissions/submissao1/results')
output_csv = output_dir / 'submission_logistic_regression.csv'

for required in [input_csv, train_csv]:
    if not required.exists():
        raise FileNotFoundError(f'Ficheiro em falta: {required}')

if model_path is None:
    raise FileNotFoundError(f'Modelo nao encontrado. Tentei: {model_candidates}')
if bow_path is None:
    raise FileNotFoundError(f'BoW nao encontrado. Tentei: {bow_candidates}')

output_dir.mkdir(parents=True, exist_ok=True)

df_sub = pd.read_csv(input_csv, sep=';')
if 'ID' not in df_sub.columns or 'Text' not in df_sub.columns:
    raise ValueError('subm1.csv deve ter colunas ID e Text')

df_train = pd.read_csv(train_csv, sep=';')
label_map = {label: i for i, label in enumerate(df_train['Label'].unique())}
idx_to_label = {i: label for label, i in label_map.items()}

df_sub['text_clean'] = df_sub['Text'].fillna('').apply(clean_text)

bow = NumpyBagOfWords()
bow.load(str(bow_path))
X_sub = bow.transform(df_sub['text_clean'])

model = LogisticRegression(n_features=len(bow.vocab), n_classes=len(label_map))
model.load(str(model_path))
pred_ids = model.predict(X_sub)

df_sub['Label'] = [idx_to_label[int(i)] for i in pred_ids]
df_out = df_sub[['ID', 'Label']]
df_out.to_csv(output_csv, sep=';', index=False)

print(f'Submissao guardada em: {output_csv}')
print(df_out.head())

Submissao guardada em: ..\submissions\submissao1\results\submission_logistic_regression.csv
     ID   Label
0  D2-1  OpenAI
1  D2-2  OpenAI
2  D2-3  OpenAI
3  D2-4  OpenAI
4  D2-5  OpenAI
